# Homework: Advanced Computer Vision

Welcome to your final Computer Vision assignment! Today, you will prove your mastery over the two dominant branches of modern CV: **Semantic Segmentation** and **Object Detection**.

**Your Mission:**
1. **Part 1: Build U-Net from Scratch.** You will implement the famous U-Net architecture for Semantic Segmentation.
2. **Part 2: YOLO Fine-Tuning & Video Inference.** You will use a state-of-the-art YOLO model, fine-tune it, and run it on a real-world video.

## Part 1: U-Net (The "Tensor Tetris" Challenge)

**EXPLICIT WARNING REGARDING AI**

Do **NOT** use AI to write this U-Net code.

Why? Because an AI will output the correct code in 0.5 seconds, and you will learn absolutely nothing. The entire point of this exercise is to struggle with "Tensor Tetris"—the art of keeping track of spatial dimensions, channel counts, and concatenations in your head.

If you don't fight with the shape mismatches here, you will be completely lost when you have to debug a real neural network on the job. Use documentation, use your notes, but write the code yourself.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### 1.1 The Double Convolution Block
In U-Net, every step of the encoder and decoder features two consecutive 3x3 Convolutions, each followed by a ReLU activation. (Modern implementations also use BatchNorm).

**Task:** Implement the `DoubleConv` block.
* `Conv2d` -> `BatchNorm2d` -> `ReLU` -> `Conv2d` -> `BatchNorm2d` -> `ReLU`
* Make sure `padding=1` in your Conv2d layers so the spatial dimensions don't shrink!

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # TODO: Define the sequential block described above
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

### 1.2 The U-Net Architecture
Now, assemble the U-Net.

**The Encoder (Down):**
* Uses your `DoubleConv`.
* Uses `nn.MaxPool2d(2)` to halve the spatial dimensions.

**The Decoder (Up):**
* Uses `nn.ConvTranspose2d` to double the spatial dimensions.
* Uses `torch.cat` to concatenate the skip connection from the encoder.
* Uses your `DoubleConv` to process the concatenated feature map.


In [4]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # --- ENCODER (Downsampling) ---
        # Image goes from 3 channels -> 64 channels
        self.inc = DoubleConv(in_channels, 64)

        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        # The Bottleneck (512 -> 1024)
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # --- DECODER (Upsampling) ---
        # TODO: Define the upsampling layers.
        # HINT: The ConvTranspose2d cuts the channels in half (e.g. 1024 -> 512).
        # Then, you concatenate a skip connection of 512 channels, bringing the total back to 1024.
        # Finally, the DoubleConv processes that 1024 back down to 512.

        self.up1_transpose = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up1_conv = DoubleConv(1024, 512)

        self.up2_transpose = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up2_conv = DoubleConv(512, 256)

        self.up3_transpose = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up3_conv = DoubleConv(256, 128)

        self.up4_transpose = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up4_conv = DoubleConv(128, 64)

        # --- FINAL OUTPUT ---
        # A simple 1x1 convolution to map the channels to the number of classes
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # ENCODER
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)  # This is the bottleneck

        # DECODER
        # Step 1: Upsample the bottleneck
        x_up = self.up1_transpose(x5)
        # Step 2: Concatenate with the corresponding skip connection (x4)
        x_concat = torch.cat([x_up, x4], dim=1)
        # Step 3: Pass through the double convolution
        x_out = self.up1_conv(x_concat)

        x_up = self.up2_transpose(x_out)
        x_concat = torch.cat([x_up, x3], dim=1)
        x_out = self.up2_conv(x_concat)

        x_up = self.up3_transpose(x_out)
        x_concat = torch.cat([x_up, x2], dim=1)
        x_out = self.up3_conv(x_concat)

        x_up = self.up4_transpose(x_out)
        x_concat = torch.cat([x_up, x1], dim=1)
        x_out = self.up4_conv(x_concat)

        logits = self.outc(x_out)

        return logits


### 1.3 The U-Net Sanity Check
If you wired your network correctly, this cell will run without error and output a tensor of shape `[1, 2, 256, 256]`.
If it crashes, read the error carefully. It will tell you exactly which layers have mismatched channel counts or spatial dimensions!


In [5]:
print("Testing U-Net Architecture...")
model = UNet(in_channels=3, num_classes=2)
dummy_image = torch.randn(1, 3, 256, 256)

try:
    output = model(dummy_image)
    print(f"Success! Output shape: {output.shape}")
    assert output.shape == (1, 2, 256, 256), "Output shape is incorrect!"
except Exception as e:
    print(f"FAILED! Error: {e}")


Testing U-Net Architecture...
Success! Output shape: torch.Size([1, 2, 256, 256])


## Part 2: YOLO Fine-Tuning & Video Inference

Writing architectures from scratch is for academia. In the real world, you use `Ultralytics`.
In this section, you will install the Ultralytics library, fine-tune a YOLOv8 model on a toy dataset, and then use it to track objects in a video.

In [7]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00


In [8]:
# NOTE: If you are running this locally, you may need to run `!pip install ultralytics` in a separate cell first.
import urllib.request
import os

try:
    from ultralytics import YOLO
    print("Ultralytics YOLO is installed and ready!")
except ImportError:
    print("Please install ultralytics: !pip install ultralytics")

# Download a sample traffic video for inference
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
video_path = "traffic_video.mp4"

if not os.path.exists(video_path):
    print("Downloading sample video...")
    urllib.request.urlretrieve(video_url, video_path)
    print("Download complete.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics YOLO is installed and ready!


### 2.1 Fine-Tune YOLOv8

**Task:** Load a pre-trained `yolov8n.pt` (Nano) model. Train it for exactly **3 epochs** on the `coco8.yaml` dataset (a tiny, built-in dataset containing 8 images just for testing code pipelines).

*Hint: This requires exactly two lines of code using the Ultralytics API.*

In [9]:
# TODO: 1. Initialize the YOLO model with 'yolov8n.pt'
model_yolo = YOLO('yolov8n.pt')

# TODO: 2. Call the train() method on your model.
# Arguments to pass: data='coco8.yaml', epochs=3, imgsz=640
model_yolo.train(data='coco8.yaml', epochs=3, imgsz=640)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7899a09500b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

### 2.2 Inference on a Video

Object detection models don't just process static images. They process videos frame-by-frame.

**Task:** Use your fine-tuned model to run predictions on `traffic_video.mp4`.
You want to **save** the output so you can watch the video with the bounding boxes drawn on it!

*Hint:* Use the `predict()` method on your model. Look up the documentation on how to pass `source` and `save=True`.


In [10]:
print(f"Running inference on {video_path}...")

# TODO: Run the model's predict method on the video_path.
# Make sure to tell it to save the output!
results = model_yolo.predict(source=video_path, save=True)


print("Inference complete! Check the 'runs/detect/predict' folder in your directory to watch your annotated video.")

Running inference on traffic_video.mp4...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/377) /content/traffic_video.mp4: 384x640 (no detections), 375.1ms
video 1/1 (frame 2/377) /content/traffic_video.mp4: 384x640 (no detections), 429.3ms
video 1/1 (frame 3/377) /content/traffic_video.mp4: 384x640 (no detections), 285.0ms
video 1/1 (frame 4/377) /content/traffic_video.mp4: 384x640 (no detections), 297.7ms
video 1/1 (frame 5/377) /content/traffic_video.mp4: 384x640 (no detections),